# Vežba 9: Neuronske mreže

Veštački neuron, mreža sa jednim skrivenim slojem računata ručno, XOR, mreže u bibliotekama scikit-learn i PyTorch, rano zaustavljanje i regularizacija

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba9.html](https://isatvss.surge.sh/vezba9.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Veštački neuron

### Zadatak 1. Izlaz jednog neurona

Za ulaz x = (0,5; −1; 2), težine w = (0,8; 0,4; −0,3) i pomeraj b = 0,1 izračunati z i izlaz neurona sa sigmoidnom aktivacijom.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


x = np.array([0.5, -1.0, 2.0])
w = np.array([0.8, 0.4, -0.3])
b = 0.1

sabirci = " + ".join(f"({wi})·({xi})" for wi, xi in zip(w, x))
z = w @ x + b
print(f"z = w · x + b = {sabirci} + {b} = {z:.2f}")
print(f"a = g(z) = {sigmoid(z):.4f}")

## Mreža sa jednim skrivenim slojem

### Zadatak 2. Mreža za XOR ručno

Sastaviti mrežu sa dva skrivena neurona koja računa XOR: prvi skriveni neuron računa OR, drugi NAND, a izlazni neuron AND nad njima. Izračunati vrednosti skrivenog sloja i izlaz za sva četiri ulaza.

Na vežbi 5 pokazano je da jedan neuron ne može da računa XOR, jer jedna prava ne razdvaja te tačke. Skriveni sloj prevodi ulaze u novi prostor, (OR, NAND), u kome su tačke linearno razdvojive, pa ih izlazni neuron razdvaja jednom pravom. To je suština skrivenih slojeva: oni sami prave obeležja koja izlaznom sloju olakšavaju zadatak.

In [ ]:
ulazi = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

W1_xor = np.array([[20, 20], [-20, -20]])
b1_xor = np.array([-10, 30])
W2_xor = np.array([20, 20])
b2_xor = -30

skriveni = sigmoid(ulazi @ W1_xor.T + b1_xor)
izlaz = sigmoid(skriveni @ W2_xor + b2_xor)

tabela = pd.DataFrame(
    {
        "x1": ulazi[:, 0],
        "x2": ulazi[:, 1],
        "a1 (OR)": skriveni[:, 0].round(3),
        "a2 (NAND)": skriveni[:, 1].round(3),
        "h(x)": izlaz.round(3),
        "XOR": (izlaz >= 0.5).astype(int),
    }
)
print(tabela.to_string(index=False))

### Zadatak 3. Oblici matrica

Za mrežu sa tri ulaza, četiri skrivena neurona i jednim izlazom, sa nasumičnim težinama, izračunati propagaciju unapred, ispisati oblike svih matrica i prebrojati parametre.

In [ ]:
X_primer = np.array([[0.2, 0.7, 0.1]])
rng = np.random.default_rng(42)
W1 = rng.normal(size=(4, 3))
b1 = np.zeros(4)
W2 = rng.normal(size=(1, 4))
b2 = np.zeros(1)

a1 = sigmoid(X_primer @ W1.T + b1)
h = sigmoid(a1 @ W2.T + b2)
print(
    "x:",
    X_primer.shape,
    " W1:",
    W1.shape,
    " a1:",
    a1.shape,
    " W2:",
    W2.shape,
    " h:",
    h.shape,
)
print("skriveni sloj a1 =", a1.round(3))
print(f"izlaz h = {h[0, 0]:.3f}")
broj_parametara = W1.size + b1.size + W2.size + b2.size
print("broj parametara:", broj_parametara)

### Zadatak 4. XOR učenjem

Naučiti XOR mrežama sa 1, 2 i 4 skrivena neurona i uporediti procene.

In [ ]:
from sklearn.neural_network import MLPClassifier

xor = np.array([0, 1, 1, 0])
for skrivenih in [1, 2, 4]:
    mreza = MLPClassifier(
        hidden_layer_sizes=(skrivenih,),
        activation="tanh",
        solver="lbfgs",
        max_iter=2000,
        random_state=1,
    )
    mreza.fit(ulazi, xor)
    print(
        f"{skrivenih} skrivenih neurona: procene = {mreza.predict(ulazi).tolist()}",
        end="",
    )
    print(f", tačnost = {mreza.score(ulazi, xor):.2f}")

## Višeslojni perceptron na nezgodama

### Zadatak 5. MLP i logistička regresija

Na skupu nezgoda istrenirati mrežu sa jednim skrivenim slojem od 32 neurona i ranim zaustavljanjem i uporediti je sa logističkom regresijom.

Mreža je zaustavljena posle 24 epohe i ima ROC AUC 0,832, naspram 0,826 logističke regresije. Dobitak je mali, a na vežbi 7 ansambli stabala bili su na sličnom nivou. Na tabelarnim podacima sa malo obeležja neuronske mreže retko nadmašuju ansamble stabala. Njihova prednost dolazi do izražaja kod slika, zvuka i teksta.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

nezgode = pd.read_csv("https://isatvss.surge.sh/podaci/nezgode_2025.csv")
nezgode = nezgode.drop_duplicates(subset="id_nezgode").reset_index(drop=True)
nezgode["teska"] = (nezgode["posledica"] != "Sa mat.stetom").astype(int)
vreme = pd.to_datetime(nezgode["datum_vreme"], format="%d.%m.%Y,%H:%M")
nezgode["sat"] = vreme.dt.hour
nezgode["dan_u_nedelji"] = vreme.dt.dayofweek
nezgode["mesec"] = vreme.dt.month
numericke = ["sat", "dan_u_nedelji", "mesec", "lon", "lat"]
kategoricke = ["policijska_uprava", "vrsta", "opis"]
X_tr, X_te, y_tr, y_te = train_test_split(
    nezgode[numericke + kategoricke],
    nezgode["teska"],
    test_size=0.2,
    stratify=nezgode["teska"],
    random_state=42,
)
priprema = ColumnTransformer(
    [
        (
            "num",
            make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
            numericke,
        ),
        ("kat", OneHotEncoder(handle_unknown="ignore"), kategoricke),
    ]
)
mlp = Pipeline(
    [
        ("priprema", priprema),
        (
            "model",
            MLPClassifier(
                hidden_layer_sizes=(32,),
                early_stopping=True,
                validation_fraction=0.1,
                random_state=42,
            ),
        ),
    ]
)
mlp.fit(X_tr, y_tr)
logisticka = Pipeline(
    [("priprema", priprema), ("model", LogisticRegression(max_iter=1000))]
).fit(X_tr, y_tr)
mreza = mlp.named_steps["model"]
auc_mlp = roc_auc_score(y_te, mlp.predict_proba(X_te)[:, 1])
auc_log = roc_auc_score(y_te, logisticka.predict_proba(X_te)[:, 1])
print(f"Epoha do zaustavljanja: {mreza.n_iter_}")
print(f"MLP, 32 skrivena neurona: ROC AUC = {auc_mlp:.3f}")
print(f"Logistička regresija:     ROC AUC = {auc_log:.3f}")

### Zadatak 6. Preterana utreniranost mreže

Istrenirati veliku mrežu sa dva skrivena sloja od po 256 neurona, jednom bez zaštite od preterane utreniranosti, a jednom sa L2 regularizacijom i ranim zaustavljanjem, pa uporediti ROC AUC na treningu i testu.

Bez zaštite mreža ima AUC 0,967 na treningu, a samo 0,780 na testu, lošije od logističke regresije. Sa regularizacijom i ranim zaustavljanjem razlika se smanjuje na 0,874 naspram 0,824. To je ista slika preterane utreniranosti kao kod polinoma na vežbi 4 i stabala na vežbi 7, a lekovi su isti: regularizacija, manje parametara, više podataka i rano zaustavljanje.

In [ ]:
bez_zastite = dict(alpha=1e-6, early_stopping=False, max_iter=60)
sa_zastitom = dict(alpha=1e-2, early_stopping=True)
for naziv, parametri in [
    ("velika mreža, bez zaštite", bez_zastite),
    ("velika mreža, L2 i rano zaustavljanje", sa_zastitom),
]:
    model = Pipeline(
        [
            ("priprema", priprema),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(256, 256), random_state=42, **parametri
                ),
            ),
        ]
    )
    model.fit(X_tr, y_tr)
    auc_tr = roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1])
    auc_te = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    print(f"{naziv:38s} AUC trening = {auc_tr:.3f}, test = {auc_te:.3f}")

## Mreža u PyTorch-u

### Zadatak 7. Petlja učenja sa ranim zaustavljanjem

Napisati mrežu sa skrivenim slojem od 64 neurona, ReLU aktivacijom i dropout-om, pa je učiti u paketima od po 256 nezgoda. Posle svake epohe izmeriti gubitak na delu za učenje i na validaciji, sačuvati najbolji model i prekinuti učenje kada se validacija pet epoha ne popravi.

Učenje je stalo posle 11 epoha, a sačuvan je model iz 6. epohe. ROC AUC na testu je 0,831, isti kao kod mreže iz scikit-learn. Razlika nije u rezultatu, nego u kontroli: u PyTorch-u se svaki deo učenja može izmeniti.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
X_tr_np = priprema.fit_transform(X_tr).toarray().astype(np.float32)
X_te_np = priprema.transform(X_te).toarray().astype(np.float32)
y_tr_np = y_tr.to_numpy(dtype=np.float32)

granica = int(0.9 * len(X_tr_np))
X_uc, y_uc = torch.tensor(X_tr_np[:granica]), torch.tensor(y_tr_np[:granica])
X_val, y_val = torch.tensor(X_tr_np[granica:]), torch.tensor(y_tr_np[granica:])

model_t = nn.Sequential(
    nn.Linear(X_uc.shape[1], 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1)
)
gubitak = nn.BCEWithLogitsLoss()
optimizator = torch.optim.Adam(model_t.parameters(), lr=1e-3, weight_decay=1e-4)

istorija = []
najbolji_gubitak, najbolje_stanje, bez_napretka = np.inf, None, 0
for epoha in range(100):
    model_t.train()
    redosled = torch.randperm(len(X_uc))
    for i in range(0, len(X_uc), 256):
        paket = redosled[i : i + 256]
        optimizator.zero_grad()
        greska = gubitak(model_t(X_uc[paket]).squeeze(1), y_uc[paket])
        greska.backward()
        optimizator.step()
    model_t.eval()
    with torch.no_grad():
        g_uc = gubitak(model_t(X_uc).squeeze(1), y_uc).item()
        g_val = gubitak(model_t(X_val).squeeze(1), y_val).item()
    istorija.append((g_uc, g_val))
    if g_val < najbolji_gubitak - 0.0001:
        najbolji_gubitak, bez_napretka = g_val, 0
        najbolje_stanje = {k: v.clone() for k, v in model_t.state_dict().items()}
    else:
        bez_napretka += 1
        if bez_napretka == 5:
            break

model_t.load_state_dict(najbolje_stanje)
model_t.eval()
with torch.no_grad():
    p_t = torch.sigmoid(model_t(torch.tensor(X_te_np)).squeeze(1)).numpy()
print(f"Epoha: {len(istorija)}, najbolja validacija u epohi {len(istorija) - 5}")
print(f"PyTorch mreža: ROC AUC na testu = {roc_auc_score(y_te, p_t):.3f}")

### Zadatak 8. Kriva gubitka

Nacrtati gubitak na delu za učenje i na validaciji po epohama i označiti epohu sačuvanog modela.

In [ ]:
istorija_np = np.array(istorija)
plt.figure(figsize=(7, 4))
plt.plot(
    range(1, len(istorija_np) + 1),
    istorija_np[:, 0],
    "-o",
    color="#5b3cc4",
    label="učenje",
)
plt.plot(
    range(1, len(istorija_np) + 1),
    istorija_np[:, 1],
    "-o",
    color="#d9480f",
    label="validacija",
)
plt.axvline(len(istorija_np) - 5, color="gray", linestyle="--", label="sačuvani model")
plt.xlabel("Epoha")
plt.ylabel("Gubitak (log-loss)")
plt.legend()
plt.show()
print(
    pd.DataFrame(istorija_np, columns=["učenje", "validacija"])
    .round(4)
    .tail(6)
    .to_string()
)

## Zadaci za samostalan rad

### Zadatak 9. Propagacija unapred ručno

Mreža ima ulaz x = (1, 0), skriveni sloj sa težinama W1 = [[0,5; −1]; [1,5; 0,5]] i pomerajima b1 = (0; −1), i izlazni neuron sa težinama W2 = (2; −1) i pomerajem b2 = 0,5. Sve aktivacije su sigmoidne. Ručno izračunati z i a skrivenog sloja i izlaz mreže, pa rezultat proveriti kodom.

### Zadatak 10. Broj skrivenih neurona

Uporediti ROC AUC na testu za mreže sa 2, 8, 32 i 128 skrivenih neurona, sve sa ranim zaustavljanjem. Od koje veličine dalji neuroni prestaju da pomažu?

### Zadatak 11. Aktivacione funkcije

Nacrtati sigmoid, tanh i ReLU i izračunati ih za z = −2, 0 i 2. Po čemu se ReLU razlikuje od druge dve funkcije za velike pozitivne vrednosti?

### Zadatak 12. Mreža za regresiju

Na podacima o vazduhu sa vežbe 4 uporediti `MLPRegressor` sa slojevima od 32 i 16 neurona i Ridge regresiju, sa istim obeležjima i vremenskom podelom.

### Zadatak 13. XNOR iz iste mreže

Promeniti samo težine i pomeraj izlaznog neurona mreže iz zadatka 2 tako da mreža računa XNOR, negaciju funkcije XOR. Proveriti izlaz za sva četiri ulaza.